[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/begelb/latent_dynamics/blob/paper/notebooks/02_leslie3d_example1.ipynb)

In [ ]:
# On Colab, install the CMGDB fork (a prebuilt wheel; not on PyPI)
# and the paper package. Running locally uses the project venv as is.
import sys

if "google.colab" in sys.modules:
    !pip install -q cmgdb==1.3.3+fork.2 --find-links https://github.com/bernardorivas/CMGDB/releases/expanded_assets/v1.3.3+fork.2
    !pip install -q git+https://github.com/begelb/latent_dynamics.git@paper

# Section 5.2.1 - Three-dimensional Leslie: a spurious attractor

## What this notebook shows

The three-generation **Leslie population model** (paper section 5.2), a
genuinely three-dimensional system, studied through a **two-dimensional** latent
model. This is the *cautionary tale*: the latent Morse graph comes out
**tristable** (three minimal nodes), but the true system has only two stable
period-four orbits -- one latent attractor corresponds to no attractor of the
real dynamics.

This is not just numerical noise: the semiconjugacy-error bound required by the
main theorem is *violated* at that node. A small training loss does **not** by
itself guarantee that latent attractors are real. (Example 2, in the next
notebook, is the success case.)

### How to run

Edit the **parameters** cell below, then *Run All*. Three modes, run in
stages: model, training curves, Morse graph.

| `MODE` | what it does | cost |
|--------|--------------|------|
| `"replay"` | re-render the paper's saved Morse graph and Morse sets | seconds |
| `"morse"` | recompute the Morse graph of the *saved* model at your `SUBDIV` | seconds-minutes |
| `"retrain"` | train a fresh model with your `OVERRIDES`, then compute its Morse graph at `SUBDIV` | minutes-hours |

**Toy subdivisions are a qualitative preview.** Coarse CMGDB grids can merge
nearby recurrent sets and change the Morse graph; the paper figures use the
config's (finer) values. The paper value for this example is noted in the
parameters cell.

In [ ]:
# ===== PARAMETERS  (edit, then Run All) ====================================
MODE = "replay"            # "replay" | "morse" | "retrain"
SEED = None                # None -> the config's default seed
SUBDIV = (10, 14, 20)      # MODE="morse"/"retrain": (subdiv_init, subdiv_min, subdiv_max)
                           # paper value: (23, 23, 27)
OVERRIDES = {}             # MODE="retrain": config overrides (pydantic-validated), e.g.
                           #   {"training": {"epochs": 300}, "cmgdb": {"subdiv_max": 20}}
MORSE_OVERRIDES = {}       # extra cmgdb fields for the Morse cell, e.g.
                           #   {"compute_roa": True} (slow) or {"padding": False}
BOX_SCALE = "auto"         # Morse-set box size: "auto" | float | {label: float}
# ===========================================================================

## Model

Load the paper's trained model, or train a fresh one.

In [ ]:
from latentdynamics.replay import load_experiment, retrain

REPLAY_CONFIG  = "leslie3d_example1_replay"
RETRAIN_CONFIG = "leslie3d_example1"

if MODE in ("replay", "morse"):
    exp = load_experiment(REPLAY_CONFIG, seed=SEED)
elif MODE == "retrain":
    # Training only. CMGDB runs further down, so a long training run survives a
    # Morse computation that needs a different grid.
    exp = retrain(
        RETRAIN_CONFIG,
        seed=SEED,
        overrides=OVERRIDES,
        stages=("data", "scale", "train", "diagnose"),
    )
else:
    raise ValueError(f"unknown MODE {MODE!r}")
exp

## Training curves

Total loss and its terms, per epoch. In `replay` and `morse` mode these are the
saved curves of the run the paper reports; in `retrain` mode they are the run
that just finished. Training stops early when validation loss stalls, so the
curves usually end well before the configured epoch budget.

In [ ]:
import json

import matplotlib.pyplot as plt

history_path = exp.seed_dir / "logs" / "history.json"
if not history_path.exists():
    print(f"no training history at {history_path}")
else:
    history = json.loads(history_path.read_text())
    terms = [k for k in history["train"] if k != "loss_total"]

    fig, (left, right) = plt.subplots(1, 2, figsize=(11, 3.8))
    left.semilogy(history["train"]["loss_total"], label="train")
    left.semilogy(history["val"]["loss_total"], label="validation")
    left.set(xlabel="epoch", ylabel="total loss")
    left.legend()

    for term in terms:
        right.semilogy(history["val"][term], label=term)
    right.set(xlabel="epoch", ylabel="validation loss by term")
    right.legend(fontsize="small")

    fig.tight_layout()
    plt.show()
    print(f"{len(history['train']['loss_total'])} epochs")

## Morse graph computation

`replay` re-renders the paper's saved Morse graph. `morse` and `retrain` both
compute one at `SUBDIV`, so the grid is chosen here rather than inherited from
the training config.

Grid size is the binding constraint. CMGDB's cached map graph is bounded by
`CMGDB_MAPGRAPH_MAX_VERTICES`, `2**24` cells by default. A paper-resolution
grid can exceed that -- the two-dimensional contraction example needs `2**27` --
and the run then stops rather than silently falling back to a per-cell map
callback, which is orders of magnitude slower. Raising the limit costs roughly
8 bytes per cell plus 8 bytes per edge, so paper resolution wants a
large-memory machine. On a Colab runtime, keep `SUBDIV` small.

In [ ]:
if MODE == "replay":
    print("replay: using the paper's saved Morse graph")
else:
    exp = exp.recompute_morse(subdiv=SUBDIV, cmgdb_overrides=MORSE_OVERRIDES)
exp

## Morse graph

Three minimal nodes -- one of them spurious.

In [ ]:
exp.show_morse_graph()

## Morse sets

In [ ]:
exp.show_morse_sets(box_scale=BOX_SCALE)

## Latent trajectories of the two 4-periodic orbits

Each true period-four orbit is encoded into the latent space and pushed forward
under the latent map. The orbit inside the spurious Morse set never settles --
that is the failure this example exposes. (2-D latent only; shown for the
replay/morse model.)

In [ ]:
from latentdynamics.cli.render import LESLIE3D_PERIODIC_PTS

exp.show_latent_trajectory(LESLIE3D_PERIODIC_PTS, steps=4)

## Run provenance

In [ ]:
exp.diagnostics()